Імпорти та завантаження датасету

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import timeit

def load_power_data():
    df_power = pd.read_csv('data/household_power_consumption.txt', sep=';', 
                           na_values=['?'], low_memory=False)
    df_power = df_power.dropna()
    numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    for col in numeric_cols:
        df_power[col] = pd.to_numeric(df_power[col])
    return df_power

df_power = load_power_data()

Вибірки та Профілювання часу

In [ ]:
def filter_1(df):
    return df[df['Global_active_power'] > 5]

def filter_2(df):
    mask = (df['Global_intensity'] >= 19) & (df['Global_intensity'] <= 20) & \
           (df['Sub_metering_2'] > df['Sub_metering_3'])
    return df[mask]

def filter_3(df):
    sample_df = df.sample(n=500000, replace=False)
    return sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def filter_4(df):
    df_filtered = df[(df['Time'] >= '18:00:00') & (df['Global_active_power'] > 6)]
    df_filtered = df_filtered[(df_filtered['Sub_metering_2'] > df_filtered['Sub_metering_1']) & 
                              (df_filtered['Sub_metering_2'] > df_filtered['Sub_metering_3'])]
    
    half = len(df_filtered) // 2
    first_half = df_filtered.iloc[:half].iloc[2::3]
    second_half = df_filtered.iloc[half:].iloc[3::4]
    return pd.concat([first_half, second_half])

print("Час виконання фільтру 1:")
%timeit -r 3 -n 1 filter_1(df_power)
print("\nЧас виконання фільтру 2:")
%timeit -r 3 -n 1 filter_2(df_power)

Нормалізація, стандартизація, кореляція

In [ ]:
sample_data = filter_1(df_power).copy()

scaler_minmax = MinMaxScaler()
scaler_std = StandardScaler()

sample_data['GAP_normalized'] = scaler_minmax.fit_transform(sample_data[['Global_active_power']])
sample_data['GAP_standardized'] = scaler_std.fit_transform(sample_data[['Global_active_power']])

pearson_corr = sample_data['Global_active_power'].corr(sample_data['Global_intensity'], method='pearson')
spearman_corr = sample_data['Global_active_power'].corr(sample_data['Global_intensity'], method='spearman')
print(f"Pearson: {pearson_corr}, Spearman: {spearman_corr}")

sample_data['Hour'] = sample_data['Time'].str.split(':').str[0]
encoded_data = pd.get_dummies(sample_data, columns=['Hour'], drop_first=True)
display(encoded_data.head())